# 02 · Adaptive contract-review agent, Google ADK version

Same Phase 2 design as `02_agent.ipynb`, built with Google's Agent Development Kit (ADK) and GPT-5 nano through Portkey.

**How the design maps to ADK**

| Design | ADK piece |
|---|---|
| Cheap pre-pass over all six families | `LlmAgent` **prepass** (no tools), one call per contract |
| Triggers (low confidence, conflicting spans, cross-reference, undefined term, quote not found, missing) | Plain Python `triggers()`; only triggered families run the verifier |
| Adaptive loop, one action per step | `LlmAgent` **clause_verifier** with six tools: `search_contract`, `check_cross_reference`, `resolve_defined_term`, `lookup_rubric`, `ask_human`, `finalize`. ADK runs the call-tool → read-result loop |
| "Record why it chose each step" | Every tool has required `rationale` and `confidence` arguments; `after_tool_callback` logs the step |
| "Bound the loop": 4 steps per family, global tool cap, token/cost budget, no-progress detector | `before_tool_callback` blocks the call and force-finalizes; `before_model_callback` stops the model once a family is final or the budget is spent |
| Revision plan | Ordered in code by rubric §7; `LlmAgent` **revision_planner** writes the ask and fallback |

The six families of one contract run concurrently (`asyncio.gather`), each in its own ADK session.

**Three conditions for Phase 3** (`condition=` in `run_agent`):

| Condition | What runs |
|---|---|
| `rag` | Simple RAG baseline: BM25 top passages per family, one model call per family, no confidence, no loop |
| `single_pass` | Pre-pass only; triggers are recorded but the loop is disabled |
| `adaptive` | Full agent |

`results.csv` keeps the pre-pass verdict next to the final one (`prepass_verdict`, `verdict_changed`), the triggers that fired, and the character spans of each quote (`evidence_spans`, for Track A token overlap with CUAD).

Tune on the **dev contracts only** (C01, C02, C11, C12). `run_agent` refuses test contracts unless `allow_test=True` (final scoring run only).

## 0. Setup

Colab secrets (🔑 on the left, notebook access on): `OPENAI_API_KEY` (the course Portkey key) and `GH_TOKEN` (to clone the private repo for `contracts/` and `rubric_v1.md`).

In [ ]:
!pip -q install google-adk litellm rank_bm25 openai

import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    if not Path("contract-review-agent").exists():
        token = userdata.get("GH_TOKEN")
        !git clone -q https://{token}@github.com/JJJJeny/contract-review-agent.git
    DATA_DIR = Path("contract-review-agent")
else:
    DATA_DIR = Path(".")          # run from the repo folder
print(sorted(p.name for p in DATA_DIR.iterdir()))

## 1. Model (Portkey → GPT-5 nano through LiteLLM)

In [ ]:
import litellm
from google.adk.models.lite_llm import LiteLlm

litellm.suppress_debug_info = True

api_key = userdata.get("OPENAI_API_KEY") if IN_COLAB else os.environ.get("OPENAI_API_KEY")
MODEL = "@30875-fall26-openai/gpt-5-nano"
base_url = "https://api.portkey.ai/v1"

ADK_MODEL = LiteLlm(
    model=MODEL,
    api_key=api_key,
    custom_llm_provider="openai",
    api_base=base_url,
    _skip_mcp_handler=True,
)
MODEL_PRICE_KEY = "gpt-5-nano"     # key into PRICES below, for the cost budget
print("Model:", MODEL)

## 2. Settings: families, prices and loop bounds

All bounds live in `BOUNDS` so the report can cite them. Check `PRICES` against current OpenAI pricing.

In [ ]:
from dataclasses import dataclass, field

FAMILIES = ["payment", "termination", "liability_indemnity",
            "confidentiality", "renewal", "dispute_governing_law"]

# USD per 1M tokens (input, output). Check current OpenAI pricing before reporting costs.
PRICES = {
    "gpt-5-nano": (0.05, 0.40),
    "gpt-4.1-nano": (0.10, 0.40),
    "fake": (0.0, 0.0),
}


@dataclass
class Bounds:
    confidence_threshold: float = 0.75   # pre-pass confidence below this triggers the loop
    max_steps_per_clause: int = 4        # professor: "bound the loop"
    max_tool_calls_per_contract: int = 30  # global cap across all six families
    max_cost_usd_per_contract: float = 0.05
    max_tokens_per_contract: int = 250_000
    prepass_top_k: int = 4               # chunks retrieved per family for the pre-pass
    search_top_k: int = 3                # chunks returned by search_contract
    max_section_words: int = 350         # text returned by check_cross_reference



BOUNDS = Bounds()

## 3. Contract helpers: sections, cross-references, defined terms, quote check

`section_text("9.2")` and `section_text("Exhibit A")` back `check_cross_reference`; an exhibit that is only referred to and never filed returns `EXHIBIT NOT INCLUDED`. `define(term)` backs `resolve_defined_term`. `contains(quote)` is the quote check in `finalize`.

In [ ]:
import re
from dataclasses import dataclass, field
from pathlib import Path

# A numbered heading such as "9." "9.2" "12.1.3" at the start of a line or sentence,
# followed by a capital letter or "(" (so "30 days" or "Section 9.2" are not headings).
HEADING = re.compile(r'(?:(?<=\n)|(?<=^)|(?<=\. )|(?<=: ))[ \t]*(\d{1,2}(?:\.\d{1,2}){0,3})\.?[ \t]+(?=[A-Z(“"])')
ATTACHMENT = re.compile(r'\b(EXHIBIT|SCHEDULE|APPENDIX|ANNEX|Exhibit|Schedule|Appendix|Annex)\s+["“]?([A-Z0-9]{1,2})\b["”]?')
CROSS_REF = re.compile(
    r'(?:\b(?:Section|Sections|Paragraph|Clause|Article)\s+\d{1,2}(?:\.\d{1,2})*(?:\s*\([a-z0-9]{1,4}\))*'
    r'|§\s*\d{1,2}(?:\.\d{1,2})*'
    r'|\b(?:Exhibit|Schedule|Appendix|Annex|EXHIBIT|SCHEDULE)\s+["“]?[A-Z0-9]{1,2}\b["”]?'
    r'|\bnotwithstanding\b|\bexcept as (?:set forth|provided|otherwise)|\bsubject to (?:Section|Article|Paragraph|Clause|Exhibit|Schedule)\b)',
    re.I)
DEFINED = re.compile(r'[("“]\s*(?:the\s+)?["“]([A-Z][A-Za-z0-9\-]*(?:\s+[A-Z][A-Za-z0-9\-]*){0,4})["”]')
MEANS = re.compile(r'\b([A-Z][A-Za-z\-]*(?:\s+[A-Z][A-Za-z\-]*){0,4})["”]?\s+(?:shall\s+)?(?:means?|has the meaning)\b')
ROLE_TERMS = {"Customer", "Provider", "Recipient", "Service", "Services", "Company", "Contractor", "Client", "Vendor"}
TRIVIAL_TERMS = {"Agreement", "Party", "Parties", "Effective Date", "Section", "Exhibit", "Schedule"}


def norm(s) -> str:
    s = s or ""
    return re.sub(r"\s+", " ", s.replace("“", '"').replace("”", '"').replace("’", "'")).strip()


@dataclass
class Chunk:
    idx: int
    start: int
    end: int
    section: str
    text: str


@dataclass
class Contract:
    cid: str
    text: str
    meta: dict = field(default_factory=dict)
    headings: list = field(default_factory=list)   # [(pos, "9.2")]
    chunks: list = field(default_factory=list)
    defined_terms: dict = field(default_factory=dict)  # term -> definition sentence
    preamble_terms: set = field(default_factory=set)

    @classmethod
    def load(cls, path, meta=None, chunk_words=110, overlap=30):
        text = Path(path).read_text(encoding="utf-8", errors="replace")
        c = cls(cid=Path(path).stem, text=text, meta=meta or {})
        c.headings = [(m.start(1), m.group(1)) for m in HEADING.finditer(text)]
        c._chunk(chunk_words, overlap)
        c._index_terms()
        return c

    def section_at(self, pos: int) -> str:
        sec = ""
        for p, s in self.headings:
            if p > pos:
                break
            sec = s
        return f"§{sec}" if sec else "untitled"

    def _chunk(self, size, overlap):
        words = [(m.start(), m.end()) for m in re.finditer(r"\S+", self.text)]
        step = size - overlap
        for i in range(0, max(1, len(words)), step):
            span = words[i:i + size]
            if not span:
                break
            s, e = span[0][0], span[-1][1]
            self.chunks.append(Chunk(len(self.chunks), s, e, self.section_at(s), norm(self.text[s:e])))
            if i + size >= len(words):
                break

    def _sentence_around(self, pos: int, max_chars=600) -> str:
        left = max(self.text.rfind(". ", 0, pos), self.text.rfind("\n\n", 0, pos))
        right = self.text.find(". ", pos)
        right = len(self.text) if right == -1 else right + 1
        return norm(self.text[left + 1:right])[:max_chars]

    def _index_terms(self):
        for rx in (DEFINED, MEANS):
            for m in rx.finditer(self.text):
                term = m.group(1).strip()
                if term in TRIVIAL_TERMS or len(term) < 3 or term in self.defined_terms:
                    continue
                self.defined_terms[term] = self._sentence_around(m.start())
                if m.start() < 1500:          # parties and roles are introduced in the preamble
                    self.preamble_terms.add(term)

    def contains(self, quote: str) -> bool:
        """verify_quote: does this exact text (whitespace-normalised) appear in the contract?"""
        q = norm(quote or "").strip(' ."')
        return bool(q) and q.lower() in norm(self.text).lower()

    def locate(self, quote: str):
        """Character span [start, end) of a quote in the original text (whitespace-insensitive), or None."""
        q = norm(quote).strip(' ."')
        if not q:
            return None
        if not hasattr(self, "_flat"):
            # normalised copy of the text plus a map from each normalised character to its original position
            flat, pos, prev_space = [], [], False
            for i, ch in enumerate(self.text.replace("“", '"').replace("”", '"').replace("’", "'")):
                if ch.isspace():
                    if not prev_space and flat:
                        flat.append(" ")
                        pos.append(i)
                    prev_space = True
                else:
                    flat.append(ch)
                    pos.append(i)
                    prev_space = False
            self._flat, self._pos = "".join(flat).lower(), pos
        k = self._flat.find(q.lower())
        if k < 0:
            return None
        return [self._pos[k], self._pos[k + len(q) - 1] + 1]

    def section_text(self, section_id: str, max_words=350) -> dict:
        sid = section_id.strip()
        att = ATTACHMENT.search(sid)
        if att or re.match(r'(?i)(exhibit|schedule|appendix|annex)', sid):
            return self._attachment_text(sid, max_words)
        num = re.search(r'\d{1,2}(?:\.\d{1,2})*', sid)
        if not num:
            return {"found": False, "note": f"could not parse section id {section_id!r}"}
        num = num.group(0)
        hits = [i for i, (p, s) in enumerate(self.headings) if s == num]
        if not hits:
            return {"found": False, "section": num,
                    "note": "no heading with this number; the reference may be wrong or the section may be in a missing exhibit"}
        i = hits[0]
        start = self.headings[i][0]
        depth = num.count(".")
        end = len(self.text)
        for p, s in self.headings[i + 1:]:
            if s.count(".") <= depth:
                end = p
                break
        words = norm(self.text[start:end]).split()
        return {"found": True, "section": f"§{num}", "text": " ".join(words[:max_words]),
                "truncated": len(words) > max_words}

    def _attachment_text(self, sid, max_words):
        m = re.search(r'(?i)(exhibit|schedule|appendix|annex)\s+["“]?([A-Z0-9]{1,2})', sid)
        if not m:
            return {"found": False, "note": f"could not parse attachment id {sid!r}"}
        kind, label = m.group(1).lower(), m.group(2).upper()
        rx = re.compile(rf'(?i)\b{kind}\s+["“]?{re.escape(label)}\b["”]?')
        mentions, body_starts = [], []
        for h in rx.finditer(self.text):
            before = self.text[max(0, h.start() - 40):h.start()]
            ctx = norm(self.text[max(0, h.start() - 80):h.end() + 80])
            mentions.append(ctx)
            # Treat it as the attachment itself when written as a heading (upper case, or on its own line).
            after = self.text[h.end():h.end() + 3]
            is_heading = self.text[h.start():h.end()].isupper() or before.rstrip(" \t").endswith("\n")
            # Attachments are filed after the body, so ignore table-of-contents lines ("Schedule A: Fees").
            past_body = not self.headings or h.start() > self.headings[-1][0]
            if (is_heading and past_body and not after.lstrip().startswith(":")
                    and not re.search(r'(?i)(in|to|on|of|per|under|the|see)\s*$', before)):
                body_starts.append(h.start())
        if body_starts:
            start = body_starts[-1]
            words = norm(self.text[start:]).split()
            return {"found": True, "section": f"{kind.title()} {label}", "text": " ".join(words[:max_words]),
                    "truncated": len(words) > max_words}
        return {"found": False, "section": f"{kind.title()} {label}",
                "note": "EXHIBIT NOT INCLUDED: only references to it appear in the filed text",
                "mentions": mentions[:3]}

    def define(self, term: str) -> dict:
        t = term.strip(' "“”')
        for k, v in self.defined_terms.items():
            if k.lower() == t.lower():
                return {"found": True, "term": k, "definition": v}
        # fallback: first sentence where the term appears in quotes or followed by "means"
        m = re.search(rf'["“]{re.escape(t)}["”]|{re.escape(t)}\s+(?:shall\s+)?means', self.text, re.I)
        if m:
            return {"found": True, "term": t, "definition": self._sentence_around(m.start())}
        return {"found": False, "term": t, "note": "no definition found; the term may be defined in a missing exhibit"}


def cross_refs(text: str) -> list:
    return sorted({norm(m.group(0)) for m in CROSS_REF.finditer(text)})


def undefined_terms(text: str, contract: Contract, already_resolved=()) -> list:
    """Defined terms used in a span whose definition is not yet part of the evidence."""
    out = []
    for term in contract.defined_terms:
        if term in already_resolved:
            continue
        if term in contract.preamble_terms or term in ROLE_TERMS:
            continue
        if re.search(rf'\b{re.escape(term)}\b', text):
            out.append(term)
    return out

## 4. Retrieval for `search_contract`: sparse (BM25), dense (embeddings) or hybrid

In [ ]:
import re
import numpy as np
from rank_bm25 import BM25Okapi

TOKEN = re.compile(r"[a-z0-9]+")


def tok(s):
    return TOKEN.findall(s.lower())


class Retriever:
    def __init__(self, contract, embedder=None):
        self.c = contract
        self.bm25 = BM25Okapi([tok(ch.text) for ch in contract.chunks])
        self.embedder = embedder          # callable(list[str]) -> np.ndarray, or None
        self._vecs = None

    def _dense_scores(self, query):
        if self._vecs is None:
            self._vecs = self.embedder([ch.text for ch in self.c.chunks])
        q = self.embedder([query])[0]
        return self._vecs @ q / (np.linalg.norm(self._vecs, axis=1) * np.linalg.norm(q) + 1e-9)

    def search(self, query, mode="hybrid", k=3, exclude=()):
        """Return the top-k chunks as dicts. mode: sparse | dense | hybrid."""
        if mode in ("dense", "hybrid") and self.embedder is None:
            mode = "sparse"               # no embeddings available: degrade, and say so in the result
        sparse = self.bm25.get_scores(tok(query))
        if mode == "sparse":
            score = sparse
        else:
            dense = self._dense_scores(query)
            if mode == "dense":
                score = dense
            else:
                rank = lambda a: np.argsort(np.argsort(-a))
                score = 1 / (60 + rank(sparse)) + 1 / (60 + rank(dense))
        order = [i for i in np.argsort(-score) if i not in exclude][:k]
        return mode, [{"chunk": int(i), "section": self.c.chunks[i].section,
                       "text": self.c.chunks[i].text} for i in order]

In [ ]:
from openai import OpenAI

EMBED_MODEL = "@30875-fall26-openai/text-embedding-3-small"


def make_embedder():
    client = OpenAI(api_key=api_key, base_url=base_url)

    def embed(texts):
        out = []
        for i in range(0, len(texts), 100):
            r = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + 100])
            out += [d.embedding for d in r.data]
        return np.array(out, dtype=np.float32)

    try:
        embed(["test"])
        return embed
    except Exception as e:            # the course key may not allow embeddings: search falls back to sparse
        print("dense retrieval off:", str(e)[:200])
        return None


EMBEDDER = make_embedder()

## 5. Rubric lookup for `lookup_rubric`

In [ ]:
import re
from pathlib import Path


def slug(s):
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")


class Rubric:
    def __init__(self, path):
        self.text = Path(path).read_text(encoding="utf-8")
        self.rules = {}           # rule_id -> text
        self.family_keywords = {}
        self._parse()

    def _parse(self):
        parts = re.split(r"^(#{2,3} .+)$", self.text, flags=re.M)
        for head, body in zip(parts[1::2], parts[2::2]):
            m = re.match(r"#{2,3} (\d+(?:\.\d+)?)\.? (.+)", head)
            if not m:
                continue
            num, title = m.group(1), m.group(2).strip()
            self.rules[num] = f"{head}\n{body.strip()}"
            fam = title.split()[0]
            if re.fullmatch(r"[a-z_]+", fam):              # "## 1. payment" etc.
                self.rules[fam] = self.rules[num]
                kw = re.search(r"\*\*Keywords:\*\* (.+)", body)
                if kw:
                    self.family_keywords[fam] = kw.group(1).rstrip(".")
                for row in re.findall(r"^\| ([^|]+?) \| (.+)$", body, flags=re.M):
                    name = row[0].strip()
                    if name.startswith("Sub-criterion") or set(name) <= set("-"):
                        continue
                    self.rules[f"{fam}.{slug(name)}"] = f"| {name} | {row[1]}"
                for label in ("Missing", "Severity", "Absence rule", "Customer is not the payer"):
                    p = re.search(rf"^\*\*{label}[^*]*\*\*.*$", body, flags=re.M)
                    if p:
                        self.rules[f"{fam}.{slug(label)}"] = p.group(0)

    def family_rules(self, family):
        return self.rules.get(family, "")

    def general_rules(self):
        return "\n\n".join(self.rules[k] for k in ("0.2", "0.3", "0.4") if k in self.rules)

    def lookup(self, rule_id):
        rid = rule_id.strip().lstrip("§").lower()
        if rid in self.rules:
            return {"found": True, "rule_id": rid, "text": self.rules[rid]}
        near = [k for k in self.rules if rid.split(".")[0] in k][:12]
        return {"found": False, "rule_id": rid, "available": near or sorted(self.rules)[:20]}


RUBRIC = Rubric(DATA_DIR / "rubric_v1.md")
print(len(RUBRIC.rules), "rules")

## 6. Token and cost accounting

In [ ]:
import json
import re
import time

import numpy as np



class Usage:
    def __init__(self, model):
        self.model = model
        self.input_tokens = self.output_tokens = self.calls = 0

    def add(self, i, o):
        self.input_tokens += i
        self.output_tokens += o
        self.calls += 1
        return self.cost_of(i, o)

    def cost_of(self, i, o):
        pin, pout = PRICES.get(self.model, (0.0, 0.0))
        return (i * pin + o * pout) / 1e6

    @property
    def tokens(self):
        return self.input_tokens + self.output_tokens

    @property
    def cost(self):
        return self.cost_of(self.input_tokens, self.output_tokens)


def parse_json(text):
    """Parse a model reply that should be JSON. Never raises.

    Small models sometimes wrap the JSON in prose or send several objects back to back
    ("Extra data" errors). Every object in the text is decoded; "families" / "items" are
    merged across objects, otherwise the largest object wins.
    """
    text = text or ""
    dec, objs, i = json.JSONDecoder(), [], 0
    while True:
        i = text.find("{", i)
        if i < 0:
            break
        try:
            obj, i = dec.raw_decode(text, i)
            if isinstance(obj, dict):
                objs.append(obj)
        except json.JSONDecodeError:
            i += 1
    if not objs:
        print("warning: no JSON in model reply:", text[:200].replace("\n", " "))
        return {}
    for key in ("families", "items"):
        if any(key in o for o in objs):
            merged = {} if key == "families" else []
            for o in objs:
                part = o.get(key)
                if isinstance(merged, dict) and isinstance(part, dict):
                    merged.update(part)
                elif isinstance(merged, list) and isinstance(part, list):
                    merged += part
            return {key: merged}
    return max(objs, key=lambda o: len(json.dumps(o, default=str)))

## 7. Prompts

In [ ]:
PERSPECTIVE = (
    "You review vendor service agreements for a SMALL-BUSINESS CUSTOMER that buys maintenance or "
    "services. It has no lawyer and little bargaining power. Judge every clause from the customer's "
    "side, even when the provider filed the contract. Apply the rubric exactly; do not invent thresholds."
)

PREPASS_SYSTEM = PERSPECTIVE + """

This is a quick first pass. For EACH of the six families you get a few candidate passages retrieved
from the contract. Give a first verdict, but be honest about confidence: lower it when the passages
do not show the whole clause, refer to other sections or exhibits, use defined terms you cannot see,
or point in different directions.

Rules that apply to every family:
{general}

Family rules:
{families}

Return JSON only:
{{"families": {{"<family>": {{
  "verdict": "acceptable|risky|missing|unclear",
  "severity": "high|medium|low|n/a",
  "evidence_text": "exact sentence copied from a passage (empty if missing)",
  "section_ref": "section as printed, e.g. §9.2",
  "evidence_2": "second exact quote the verdict depends on, or empty",
  "section_ref_2": "",
  "confidence": 0.0-1.0,
  "spans_conflict": true|false,
  "notes": "one or two sentences: which sub-criterion decided it"
}}}}}}
Include all six families."""


PLAN_SYSTEM = PERSPECTIVE + """

Write the negotiation ask for each finding. For each item give one plain sentence the customer can
say to the provider ("ask") and a fallback if the provider says no. Return JSON only:
{"items": [{"id": int, "ask": str, "fallback": str}]}"""

RAG_SYSTEM = PERSPECTIVE + """

Baseline: read the retrieved passages for ONE clause family and judge it with the rubric below.
Return JSON only:
{{"verdict": "acceptable|risky|missing|unclear", "severity": "high|medium|low|n/a",
  "evidence_text": "exact sentence copied from a passage, or empty", "section_ref": "",
  "evidence_2": "", "section_ref_2": "", "notes": "which sub-criterion decided it"}}

Rubric:
{rules}"""

## 8. Triggers (computed in code, never by the model)

A family enters the loop only if one of these fires.

In [ ]:
SEV_ORDER = {"high": 0, "medium": 1, "low": 2, "n/a": 3}
VERDICT_ORDER = {"risky": 0, "missing": 1, "unclear": 2, "acceptable": 3}
FAMILY_TIEBREAK = ["liability_indemnity", "termination", "payment", "renewal",
                   "confidentiality", "dispute_governing_law"]   # rubric §7 tie-break

def triggers(contract, draft, evidence, chased=(), resolved=()):
    """Why a family needs the loop. Computed in code, never by the model."""
    quotes = " ".join(e.get("text", "") for e in evidence if e.get("source") in ("prepass", "cross_ref"))
    t = {}
    if float(draft.get("confidence") or 0) < BOUNDS.confidence_threshold:
        t["low_confidence"] = draft.get("confidence")
    if draft.get("spans_conflict"):
        t["spans_conflict"] = True
    refs = [r for r in cross_refs(quotes) if r.lower() not in chased]
    if refs:
        t["cross_refs"] = refs[:5]
    terms = undefined_terms(quotes, contract, resolved)
    if terms:
        t["undefined_terms"] = terms[:5]
    for key in ("evidence_text", "evidence_2"):
        if draft.get(key) and not contract.contains(draft[key]):
            t["quote_not_found"] = key
    if draft.get("verdict") == "missing":
        t["missing_needs_search"] = True
    return t


def short(ev, n=90):
    return [f'{e.get("section", "")}: {norm(e.get("text", ""))[:n]}' for e in ev]

## 9. ADK tools: the six actions

Each tool is a plain Python function; ADK turns the signature and docstring into the tool schema. `tool_context` is filled in by ADK and gives the tool the family's session state (evidence, draft, steps).

In [ ]:
import asyncio, json, time
from google.adk.agents import LlmAgent
from google.adk.models.llm_response import LlmResponse
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import ToolContext
from google.genai import types

CTX = {}          # contract_id -> shared objects for one contract run (contract, retriever, budget, trace)


def _ctx(tc):
    return CTX[tc.state["contract_id"]]


def _add_evidence(tc, items):
    tc.state["evidence"] = tc.state.get("evidence", []) + items


def search_contract(query: str, mode: str, rationale: str, confidence: float, tool_context: ToolContext) -> dict:
    """Re-retrieve passages from the contract with a different query.

    Args:
      query: words to search for.
      mode: "sparse" (keywords), "dense" (meaning) or "hybrid" (both).
      rationale: one sentence on why this action now, naming the open trigger it addresses.
      confidence: your confidence in your current draft verdict, 0 to 1.
    """
    c = _ctx(tool_context)
    used, hits = c["retriever"].search(query, mode, BOUNDS.search_top_k)
    _add_evidence(tool_context, [{**h, "source": "search"} for h in hits])
    tool_context.state["searches"] = tool_context.state.get("searches", []) + [query]
    return {"mode_used": used, "hits": hits}


def check_cross_reference(section_id: str, rationale: str, confidence: float, tool_context: ToolContext) -> dict:
    """Read a section or exhibit that the evidence refers to, e.g. "9.2", "Section 17(b)", "Exhibit A".

    Args:
      section_id: the section number or exhibit name.
      rationale: one sentence on why this action now, naming the open trigger it addresses.
      confidence: your confidence in your current draft verdict, 0 to 1.
    """
    c = _ctx(tool_context)
    chased = set(tool_context.state.get("chased", []))
    chased.add(section_id.lower())
    for r in tool_context.state.get("open_triggers", {}).get("cross_refs", []):
        if section_id.lower().replace("section", "").strip(" §") in r.lower():
            chased.add(r.lower())
    tool_context.state["chased"] = sorted(chased)
    res = c["contract"].section_text(section_id, BOUNDS.max_section_words)
    if res.get("found"):
        _add_evidence(tool_context, [{"section": res["section"], "text": res["text"], "source": "cross_ref"}])
    return res


def resolve_defined_term(term: str, rationale: str, confidence: float, tool_context: ToolContext) -> dict:
    """Pull the definition of a capitalized defined term used in the evidence.

    Args:
      term: the term, e.g. "Fees" or "Confidential Information".
      rationale: one sentence on why this action now, naming the open trigger it addresses.
      confidence: your confidence in your current draft verdict, 0 to 1.
    """
    res = _ctx(tool_context)["contract"].define(term)
    tool_context.state["resolved"] = tool_context.state.get("resolved", []) + [term]
    if res.get("found"):
        _add_evidence(tool_context, [{"section": f'definition of "{res["term"]}"', "text": res["definition"],
                                      "source": "definition"}])
    return res


def lookup_rubric(rule_id: str, rationale: str, confidence: float, tool_context: ToolContext) -> dict:
    """Re-read one rubric rule, e.g. "payment.late_charges", "termination.cure_period" or "0.4".

    Args:
      rule_id: the rule id.
      rationale: one sentence on why this action now, naming the open trigger it addresses.
      confidence: your confidence in your current draft verdict, 0 to 1.
    """
    return _ctx(tool_context)["rubric"].lookup(rule_id)


def ask_human(question: str, rationale: str, confidence: float, tool_context: ToolContext) -> dict:
    """Queue a question for the human reviewer. Does not stop you; keep working and finalize.

    Args:
      question: the question, with the clause it is about.
      rationale: one sentence on why a human is needed.
      confidence: your confidence in your current draft verdict, 0 to 1.
    """
    q = _ctx(tool_context)["human_queue"]
    q.append({"contract_id": tool_context.state["contract_id"], "family": tool_context.state["family"],
              "question": question, "draft": tool_context.state.get("draft", {})})
    return {"queued": True, "position": len(q)}


def finalize(verdict: str, severity: str, evidence_text: str, section_ref: str, evidence_2: str,
             section_ref_2: str, notes: str, rationale: str, confidence: float, tool_context: ToolContext) -> dict:
    """Stop and record the verdict for this family.

    Args:
      verdict: acceptable, risky, missing or unclear.
      severity: high, medium, low or n/a (n/a only for acceptable).
      evidence_text: exact sentence copied from the evidence ("" or "NONE FOUND" if missing). It is checked.
      section_ref: section as printed, e.g. "§9.2".
      evidence_2: second exact quote the verdict depends on, or "".
      section_ref_2: its section, or "".
      notes: which sub-criterion decided it; for missing, the keywords and sections searched.
      rationale: one sentence on why you can stop now.
      confidence: your confidence in this verdict, 0 to 1.
    """
    c = _ctx(tool_context)["contract"]
    bad = [k for k, q in (("evidence_text", evidence_text), ("evidence_2", evidence_2))
           if q and q != "NONE FOUND" and not c.contains(q)]
    if bad and not tool_context.state.get("quote_rejected"):
        tool_context.state["quote_rejected"] = True
        return {"rejected": True, "reason": f"{', '.join(bad)} not found verbatim in the contract; copy it exactly"}
    tool_context.state["final"] = {"verdict": verdict, "severity": severity, "evidence_text": evidence_text,
                                   "section_ref": section_ref, "evidence_2": evidence_2,
                                   "section_ref_2": section_ref_2, "notes": notes, "confidence": confidence,
                                   "path": "loop"}
    return {"accepted": True}


TOOLS = [search_contract, check_cross_reference, resolve_defined_term, lookup_rubric, ask_human, finalize]
ARG_NOISE = {"rationale", "confidence"}

## 10. Callbacks: the bounds and the step log

- `before_tool_callback`: counts the step, blocks a repeated call (no progress), any non-finalize call at the step limit, and anything past the global tool cap or budget. A blocked call force-finalizes the current draft and queues a question for a human.
- `after_tool_callback`: writes the trace record `{step, trigger_reason, action, rationale, evidence_before/after, confidence_before/after, cost, elapsed}` and returns the remaining `open_triggers` and `steps_left` to the model.
- `before_model_callback`: ends the run once the family is final, stopped, or out of budget.
- `after_model_callback`: adds the call's tokens to the budget.

In [ ]:
def _usage(cb_ctx):
    return CTX[cb_ctx.state["contract_id"]]["usage"]


def over_budget(c):
    if c["tool_calls"] >= BOUNDS.max_tool_calls_per_contract:
        return "global tool-call cap"
    if c["usage"].cost >= BOUNDS.max_cost_usd_per_contract:
        return "cost budget"
    if c["usage"].tokens >= BOUNDS.max_tokens_per_contract:
        return "token budget"
    return None


def force_finalize(state, c, reason):
    """Stop the loop from code: keep the current draft, mark it forced, queue it for a human."""
    d = dict(state.get("draft", {}))
    c_ = c["contract"]
    if d.get("evidence_text") and not c_.contains(d["evidence_text"]):
        d["evidence_text"] = ""                       # never keep a quote that is not in the contract
    if d.get("evidence_2") and not c_.contains(d["evidence_2"]):
        d["evidence_2"] = ""
    if not d.get("evidence_text") and d.get("verdict") != "missing":
        for e in state.get("evidence", []):           # newest verified passage the agent actually read
            if e.get("source") != "definition" and c_.contains(e["text"]):
                d["evidence_text"], d["section_ref"] = e["text"], e["section"]
    d["notes"] = (d.get("notes") or "") + f" [forced finalize: {reason}; queued for human]"
    d["path"] = f"forced:{reason}"
    state["final"] = d
    state["stop"] = reason
    c["human_queue"].append({"contract_id": state["contract_id"], "family": state["family"],
                             "question": f"Agent stopped ({reason}). Please check this verdict.", "draft": d})


STOP = LlmResponse(content=types.Content(role="model", parts=[types.Part(text="stopped")]))


def before_model(callback_context, llm_request):
    st, c = callback_context.state, CTX[callback_context.state["contract_id"]]
    if st.get("final") or st.get("stop"):
        return STOP                                   # skip the model call: the loop is over
    if callback_context.agent_name == "clause_verifier" and \
            st.get("steps", 0) >= BOUNDS.max_steps_per_clause - 1:
        # last step: offer only finalize, so the model must decide with the evidence it has
        for t in llm_request.config.tools or []:
            if t.function_declarations:
                t.function_declarations = [f for f in t.function_declarations if f.name == "finalize"]
        llm_request.tools_dict = {k: v for k, v in llm_request.tools_dict.items() if k == "finalize"}
        llm_request.append_instructions(["This is your LAST step. Call finalize now, using the evidence you "
                                         "have already read. Copy quotes exactly."])
    reason = over_budget(c)
    if reason:
        force_finalize(st, c, reason)
        log(c, st, step=st.get("steps", 0) + 1, phase="forced", trigger_reason=f"budget: {reason}",
            action="finalize", args={}, rationale=f"{reason} reached; keeping current draft", forced=reason)
        return STOP
    return None


def after_model(callback_context, llm_response):
    u = llm_response.usage_metadata
    if u:
        i, o = u.prompt_token_count or 0, u.candidates_token_count or 0
        _usage(callback_context).add(i, o)
        # per-family running total, so each logged step gets the tokens of the call that chose it
        pi, po = callback_context.state.get("_unlogged", [0, 0])
        callback_context.state["_unlogged"] = [pi + i, po + o]
    return None


def log(c, st, **kw):
    i, o = st.get("_unlogged", [0, 0])
    st["_unlogged"] = [0, 0]
    kw.setdefault("evidence_before", short(st.get("evidence", [])))
    kw.setdefault("evidence_after", kw["evidence_before"])
    kw.setdefault("confidence_before", st.get("draft", {}).get("confidence"))
    kw.setdefault("confidence_after", kw["confidence_before"])
    kw.setdefault("elapsed", 0.0)
    c["trace"].append({"contract_id": st["contract_id"], "family": st["family"], **kw,
                       "cost": round(c["usage"].cost_of(i, o), 6), "tokens": i + o})


def before_tool(tool, args, tool_context):
    st, c = tool_context.state, CTX[tool_context.state["contract_id"]]
    # small models sometimes write "Rationale" or leave an argument out; fix it here instead of
    # letting the call fail and waste a step
    for k in list(args):
        if k.lower() != k and k.lower() in ("rationale", "confidence"):
            args[k.lower()] = args.pop(k)
    args.setdefault("rationale", "(no rationale given)")
    args.setdefault("confidence", st.get("draft", {}).get("confidence") or 0.0)
    if tool.name == "finalize":
        for k in ("evidence_2", "section_ref_2", "notes", "section_ref", "evidence_text"):
            args.setdefault(k, "")
    step = st.get("steps", 0) + 1
    st["steps"] = step
    draft = dict(st.get("draft", {}))
    conf_before = draft.get("confidence")
    if "confidence" in args:
        draft["confidence"] = args["confidence"]
    if tool.name == "finalize":
        draft.update({k: args.get(k) for k in ("verdict", "severity", "evidence_text", "section_ref", "notes")})
    st["draft"] = draft
    key = tool.name + json.dumps({k: v for k, v in args.items() if k not in ARG_NOISE}, sort_keys=True)
    seen = st.get("seen", [])
    forced = None
    if key in seen and tool.name != "finalize":
        forced = "no progress: same action and args repeated"
    elif step >= BOUNDS.max_steps_per_clause and tool.name != "finalize":
        forced = "step limit reached"
    elif over_budget(c):
        forced = over_budget(c)
    trig = ", ".join(sorted(st.get("open_triggers", {}))) or "none"
    if forced:
        force_finalize(st, c, forced)
        log(c, st, step=step, phase="forced", trigger_reason=trig, action=tool.name, args=args,
            rationale=args.get("rationale", ""), forced=forced, confidence_before=conf_before,
            confidence_after=draft.get("confidence"))
        return {"stopped": forced}                    # the tool does not run
    st["seen"] = seen + [key]
    c["tool_calls"] += 1
    st["_pending"] = {"t0": time.time(), "evidence_before": short(st.get("evidence", [])),
                      "confidence_before": conf_before, "trigger_reason": trig}
    return None


def after_tool(tool, args, tool_context, tool_response):
    st, c = tool_context.state, CTX[tool_context.state["contract_id"]]
    if st.get("stop"):                                # forced in before_tool; already logged
        return None
    p = st.get("_pending") or {}
    st["open_triggers"] = triggers(c["contract"], st.get("draft", {}), st.get("evidence", []),
                                   set(st.get("chased", [])), set(st.get("resolved", [])))
    log(c, st, step=st["steps"], phase="loop", trigger_reason=p.get("trigger_reason", ""), action=tool.name,
        args={k: v for k, v in args.items() if k != "rationale"}, rationale=args.get("rationale", ""),
        result=json.loads(json.dumps(tool_response, default=str)),
        evidence_before=p.get("evidence_before", []), evidence_after=short(st.get("evidence", [])),
        confidence_before=p.get("confidence_before"), confidence_after=st.get("draft", {}).get("confidence"),
        elapsed=round(time.time() - p.get("t0", time.time()), 2))
    if tool.name != "finalize" or tool_response.get("rejected"):
        return {**tool_response, "open_triggers": st["open_triggers"],
                "steps_left": BOUNDS.max_steps_per_clause - st["steps"]}
    return None

## 11. Agents

In [ ]:
VERIFIER_INSTRUCTION = PERSPECTIVE + """

You are checking ONE clause family of one contract because the first pass was not reliable enough.
The user message gives the family, the draft verdict, the evidence so far and the open triggers.

Call exactly ONE tool per turn, and finalize as soon as the evidence answers the rubric; you do not
have to use every step. Every tool takes a "rationale" (why this action now, naming the
trigger it resolves) and your current "confidence". Prefer the action that resolves an open trigger;
never repeat a call with the same arguments. Use mode "sparse" for exact words or section titles,
"dense" for a description of meaning; do not write OR-queries. Tool results show the remaining open_triggers and
steps_left. When steps_left is 1, call finalize. Quotes in finalize must be copied exactly from the
evidence; they are checked. A "missing" verdict needs the search record in notes. An exhibit that was
not filed makes the dependent term "unclear" (usually low).

Rubric for this family:
"""


def verifier_instruction(ctx):
    return VERIFIER_INSTRUCTION + RUBRIC.family_rules(ctx.state["family"])


def make_agents(model):
    prepass = LlmAgent(name="prepass", model=model, include_contents="default",
                       instruction=lambda ctx: PREPASS_SYSTEM.format(
                           general=RUBRIC.general_rules(),
                           families="\n\n".join(RUBRIC.family_rules(f) for f in FAMILIES)),
                       before_model_callback=before_model, after_model_callback=after_model)
    verifier = LlmAgent(name="clause_verifier", model=model, instruction=verifier_instruction, tools=TOOLS,
                        before_model_callback=before_model, after_model_callback=after_model,
                        before_tool_callback=before_tool, after_tool_callback=after_tool)
    planner = LlmAgent(name="revision_planner", model=model, instruction=lambda ctx: PLAN_SYSTEM,
                       before_model_callback=before_model, after_model_callback=after_model)
    rag = LlmAgent(name="rag_baseline", model=model,
                   instruction=lambda ctx: RAG_SYSTEM.format(rules=RUBRIC.family_rules(ctx.state["family"])),
                   before_model_callback=before_model, after_model_callback=after_model)
    return prepass, verifier, planner, rag

## 12. Orchestrator: pre-pass → triggers → verifier per family → revision plan

In [ ]:
SESSIONS = InMemorySessionService()
APP = "contract_review"


async def ask(agent, state, text):
    """Run one agent in a fresh session; return its last text and the final session state."""
    runner = Runner(app_name=APP, agent=agent, session_service=SESSIONS)
    s = await SESSIONS.create_session(app_name=APP, user_id="u", state=state)
    last = ""
    async for ev in runner.run_async(user_id="u", session_id=s.id,
                                     new_message=types.Content(role="user", parts=[types.Part(text=text)])):
        if ev.content and ev.content.parts:
            last = "".join(p.text or "" for p in ev.content.parts if getattr(p, "text", None)) or last
    s = await SESSIONS.get_session(app_name=APP, user_id="u", session_id=s.id)
    return last, s.state


CONDITIONS = ("rag", "single_pass", "adaptive")


async def review_contract(contract, model, embedder=None, condition="adaptive"):
    """condition: "rag" = simple RAG baseline (one call per family, no confidence, no loop);
    "single_pass" = pre-pass only, loop disabled; "adaptive" = full agent."""
    assert condition in CONDITIONS, condition
    cid = contract.cid
    c = CTX[cid] = {"contract": contract, "retriever": Retriever(contract, embedder), "rubric": RUBRIC,
                    "usage": Usage(MODEL_PRICE_KEY), "tool_calls": 0, "trace": [], "human_queue": []}
    prepass, verifier, planner, rag = make_agents(model)
    t0 = time.time()

    if condition == "rag":
        async def rag_family(fam):
            _, hits = c["retriever"].search(RUBRIC.family_keywords.get(fam, fam), "sparse", BOUNDS.prepass_top_k)
            text, _ = await ask(rag, {"contract_id": cid, "family": fam},
                                json.dumps({"family": fam, "passages": hits}, ensure_ascii=False))
            d = parse_json(text)
            d.setdefault("verdict", "unclear")
            c["trace"].append({"contract_id": cid, "family": fam, "step": 0, "phase": "rag", "trigger_reason": "",
                               "action": "rag_answer", "args": {}, "rationale": "baseline: retrieve then answer once",
                               "evidence_before": [], "evidence_after": short(hits), "confidence_before": None,
                               "confidence_after": None, "cost": 0.0, "tokens": 0, "elapsed": 0.0})
            return fam, {**d, "path": "rag", "steps": 0, "prepass": {}, "triggers": {}}
        results = dict(await asyncio.gather(*(rag_family(f) for f in FAMILIES)))
        return await finish(contract, c, results, planner, t0, condition)

    # 1. pre-pass: one call, all six families
    cands = {f: c["retriever"].search(RUBRIC.family_keywords.get(f, f), "hybrid", BOUNDS.prepass_top_k)[1]
             for f in FAMILIES}
    text, _ = await ask(prepass, {"contract_id": cid, "family": "*"},
                       json.dumps({"contract_id": cid, "candidates": cands}, ensure_ascii=False))
    pre = parse_json(text).get("families", {})
    if len([f for f in FAMILIES if isinstance(pre.get(f), dict)]) < len(FAMILIES):
        # incomplete answer (cut off or malformed): ask once more and keep whatever parses
        text2, _ = await ask(prepass, {"contract_id": cid, "family": "*"},
                             json.dumps({"contract_id": cid, "candidates": cands}, ensure_ascii=False))
        pre = {**parse_json(text2).get("families", {}), **{k: v for k, v in pre.items() if isinstance(v, dict)}}
        text = text + "\n--- retry ---\n" + text2
    c["trace"].append({"contract_id": cid, "family": "*", "step": 0, "phase": "prepass", "trigger_reason": "",
                       "action": "prepass", "args": {}, "rationale": "cheap extraction pass over retrieved candidates",
                       "result": {"families_parsed": sorted(f for f in FAMILIES if isinstance(pre.get(f), dict)),
                                  "raw_reply_start": text[:600]},
                       "evidence_before": [], "evidence_after": [], "confidence_before": None,
                       "confidence_after": {f: (pre.get(f) or {}).get("confidence") for f in FAMILIES},
                       "cost": round(c["usage"].cost, 6), "tokens": c["usage"].tokens, "elapsed": round(time.time() - t0, 2)})

    # 2. triggers decide which families enter the loop; 3. loop families run concurrently
    async def one_family(fam):
        d = dict(pre.get(fam) if isinstance(pre.get(fam), dict) else {"verdict": "unclear", "confidence": 0.0})
        ev = [{"section": d.get(r, ""), "text": d[k], "source": "prepass"}
              for k, r in (("evidence_text", "section_ref"), ("evidence_2", "section_ref_2")) if d.get(k)]
        st = {"contract_id": cid, "family": fam, "draft": d, "evidence": ev, "steps": 0}
        st["open_triggers"] = triggers(contract, d, ev)
        base = {"prepass": {k: d.get(k) for k in ("verdict", "severity", "confidence")},
                "triggers": st["open_triggers"]}
        if not st["open_triggers"] or condition == "single_pass":
            why = "no trigger fired" if not st["open_triggers"] else "loop disabled (single_pass condition)"
            log(c, st, step=0, phase="no_loop", trigger_reason=", ".join(sorted(st["open_triggers"])) or "none",
                action="finalize", args={}, rationale=f"{why}; pre-pass verdict kept")
            return fam, {**d, "path": "prepass", "steps": 0, **base}
        _, end = await ask(verifier, st, "State:\n" + json.dumps(
            {k: st[k] for k in ("family", "draft", "evidence", "open_triggers")} |
            {"steps_left": BOUNDS.max_steps_per_clause,
             "rubric_rule_ids": sorted(k for k in RUBRIC.rules if k.startswith(fam + "."))}, ensure_ascii=False))
        if not end.get("final"):                       # model answered in text without calling finalize
            force_finalize(end, c, "model stopped without finalize")
            log(c, end, step=end.get("steps", 0), phase="forced", trigger_reason="", action="finalize", args={},
                rationale="model ended its turn without finalize", forced="model stopped without finalize")
        return fam, {**end["final"], "steps": end.get("steps", 0), **base}

    results = dict(await asyncio.gather(*(one_family(f) for f in FAMILIES)))
    return await finish(contract, c, results, planner, t0, condition)


async def finish(contract, c, results, planner, t0, condition):
    cid = contract.cid
    for r in results.values():
        r["severity"] = "n/a" if r.get("verdict") == "acceptable" else r.get("severity", "low")
        for k in ("evidence_text", "evidence_2"):
            q = r.get(k) or ""
            r[k + "_verified"] = q in ("", "NONE FOUND") or contract.contains(q)
        # character spans of the quotes, for Track A (token overlap with CUAD answer spans)
        r["evidence_spans"] = [sp for sp in (contract.locate(r.get("evidence_text") or ""),
                                             contract.locate(r.get("evidence_2") or "")) if sp]

    # 4. revision plan: order by rubric §7 in code, the model writes ask + fallback
    plan = sorted(({"family": f, **{k: results[f].get(k, "") for k in ("verdict", "severity", "evidence_text",
                                                                          "section_ref", "notes")}}
                   for f in FAMILIES if results[f].get("verdict") != "acceptable"),
                  key=lambda x: (SEV_ORDER.get(x["severity"], 3), VERDICT_ORDER.get(x["verdict"], 3),
                                 FAMILY_TIEBREAK.index(x["family"])))
    for i, f in enumerate(plan, 1):
        f["id"] = i
    if plan and not over_budget(c):
        text, _ = await ask(planner, {"contract_id": cid, "family": "*"}, json.dumps({"findings": plan}))
        asks = {it.get("id"): it for it in parse_json(text).get("items", [])}
        for f in plan:
            f["ask"], f["fallback"] = asks.get(f["id"], {}).get("ask", ""), asks.get(f["id"], {}).get("fallback", "")
    return {"results": results, "plan": plan, "trace": c["trace"], "human_queue": c["human_queue"],
            "summary": {"contract_id": cid, "condition": condition, "llm_calls": c["usage"].calls, "tool_calls": c["tool_calls"],
                        "tokens": c["usage"].tokens, "cost_usd": round(c["usage"].cost, 5),
                        "elapsed_s": round(time.time() - t0, 2),
                        "families_looped": sum(1 for r in results.values() if r.get("path") not in ("prepass", "rag"))}}

## 13. Trace viewer (one HTML file for debugging, the report and the demo)

In [ ]:
import csv
import json
from pathlib import Path

PAGE = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Agent trace viewer</title>
<style>
:root{--bg:#fafaf8;--fg:#1d1d1b;--mut:#6b6b66;--line:#e2e1dc;--card:#fff;--risky:#b3261e;--unclear:#9a6700;--missing:#6f42c1;--ok:#1a7f37;--forced:#fff4e5;--acc:#0b57d0}
@media (prefers-color-scheme:dark){:root{--bg:#161615;--fg:#ecebe6;--mut:#9d9c95;--line:#33322f;--card:#1f1f1d;--risky:#f2867d;--unclear:#e3b341;--missing:#b89cf0;--ok:#56c26f;--forced:#3a2a12;--acc:#8ab4f8}}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--fg);font:14px/1.45 system-ui,-apple-system,Segoe UI,sans-serif}
main{max-width:1200px;margin:0 auto;padding:16px}h1{font-size:20px;margin:0 0 4px}h2{font-size:16px;margin:24px 0 8px}
.mut{color:var(--mut)}select{font:inherit;padding:4px 8px;background:var(--card);color:var(--fg);border:1px solid var(--line);border-radius:6px}
.stats{display:flex;flex-wrap:wrap;gap:8px;margin:12px 0}.stat{background:var(--card);border:1px solid var(--line);border-radius:8px;padding:8px 12px}
.stat b{display:block;font-size:18px}.wrap{overflow-x:auto}table{border-collapse:collapse;width:100%;background:var(--card)}
th,td{border-bottom:1px solid var(--line);padding:6px 8px;text-align:left;vertical-align:top}th{font-size:12px;color:var(--mut);font-weight:600}
tr.fam{cursor:pointer}tr.fam:hover{background:color-mix(in srgb,var(--acc) 8%,transparent)}tr.sel{outline:2px solid var(--acc)}
.v-risky{color:var(--risky);font-weight:600}.v-unclear{color:var(--unclear);font-weight:600}.v-missing{color:var(--missing);font-weight:600}.v-acceptable{color:var(--ok);font-weight:600}
tr.forced td{background:var(--forced)}code{font-size:12px;white-space:pre-wrap;word-break:break-word}.ev{font-size:12px;color:var(--mut);max-width:360px}
.pill{display:inline-block;border:1px solid var(--line);border-radius:10px;padding:0 6px;font-size:12px;margin:1px}
</style></head><body><main>
<h1>Agent trace viewer</h1><div class="mut" id="meta"></div>
<p>Contract <select id="pick"></select></p>
<div class="stats" id="stats"></div>
<h2>Verdicts <span class="mut">(click a row to see its steps)</span></h2><div class="wrap"><table id="res"></table></div>
<h2 id="steps-h">Steps</h2><div class="wrap"><table id="steps"></table></div>
<h2>Revision plan</h2><div class="wrap"><table id="plan"></table></div>
<h2>Questions queued for a human</h2><div class="wrap"><table id="queue"></table></div>
</main><script>
const D=__DATA__;
const esc=s=>String(s??"").replace(/[&<>"]/g,c=>({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));
const pick=document.getElementById("pick");
D.ids.forEach(id=>pick.add(new Option(id+" "+(D.rows.find(r=>r.contract_id==id)||{}).short_name,id)));
document.getElementById("meta").textContent=`model ${D.summary.model} · bounds: ${D.summary.bounds.max_steps_per_clause} steps/clause, ${D.summary.bounds.max_tool_calls_per_contract} tool calls/contract, $${D.summary.bounds.max_cost_usd_per_contract}/contract, confidence < ${D.summary.bounds.confidence_threshold} triggers the loop`;
let fam=null;
function render(){
 const id=pick.value,s=D.summary.contracts.find(c=>c.contract_id==id)||{};
 document.getElementById("stats").innerHTML=[["families looped",s.families_looped+" / 6"],["LLM calls",s.llm_calls],["tool calls",s.tool_calls],["tokens",s.tokens],["cost (USD)",s.cost_usd],["time (s)",s.elapsed_s]].map(([k,v])=>`<div class="stat"><b>${esc(v)}</b>${k}</div>`).join("");
 const rows=D.rows.filter(r=>r.contract_id==id);
 document.getElementById("res").innerHTML="<tr><th>family</th><th>verdict</th><th>severity</th><th>confidence</th><th>path</th><th>steps</th><th>quote ok</th><th>evidence</th><th>notes</th></tr>"+rows.map(r=>`<tr class="fam ${r.family==fam?"sel":""}" data-f="${r.family}"><td>${r.family}</td><td class="v-${r.verdict}">${r.verdict}</td><td>${r.severity}</td><td>${esc(r.confidence)}</td><td>${esc(r.path)}</td><td>${r.steps}</td><td>${r.evidence_verified=="True"?"✓":"✗"}</td><td class="ev">${esc(r.section_ref)} ${esc(r.evidence_text).slice(0,220)}</td><td class="ev">${esc(r.notes)}</td></tr>`).join("");
 document.querySelectorAll("tr.fam").forEach(tr=>tr.onclick=()=>{fam=tr.dataset.f;render()});
 const st=D.trace.filter(t=>t.contract_id==id&&(fam?t.family==fam||t.family=="*":true));
 document.getElementById("steps-h").textContent="Steps"+(fam?" · "+fam:" · all families");
 document.getElementById("steps").innerHTML="<tr><th>family</th><th>step</th><th>trigger</th><th>action</th><th>rationale</th><th>confidence</th><th>new evidence</th><th>cost</th><th>s</th></tr>"+st.map(t=>{const added=(t.evidence_after||[]).filter(e=>!(t.evidence_before||[]).includes(e));const res=t.result?JSON.stringify(t.result).slice(0,300):"";return `<tr class="${t.forced||t.phase=="forced"?"forced":""}"><td>${t.family}</td><td>${t.step}</td><td>${esc(t.trigger_reason).split(", ").map(x=>x?`<span class="pill">${esc(x)}</span>`:"").join("")}</td><td><b>${esc(t.action)}</b><br><code>${esc(JSON.stringify(t.args))}</code></td><td>${esc(t.rationale)}${t.forced?`<br><b>forced:</b> ${esc(t.forced)}`:""}${res?`<br><code class="mut">${esc(res)}</code>`:""}</td><td>${typeof t.confidence_after=="object"&&t.confidence_after?Object.entries(t.confidence_after).map(([k,v])=>k+": "+v).join("<br>"):esc(t.confidence_before)+" → "+esc(t.confidence_after)}</td><td class="ev">${added.map(esc).join("<br>")}</td><td>${t.cost}</td><td>${t.elapsed}</td></tr>`}).join("");
 const p=D.plans[id]||[];
 document.getElementById("plan").innerHTML="<tr><th>#</th><th>family</th><th>finding</th><th>evidence</th><th>ask</th><th>fallback</th></tr>"+p.map(f=>`<tr><td>${f.id}</td><td>${f.family}</td><td class="v-${f.verdict}">${f.verdict} · ${f.severity}</td><td class="ev">${esc(f.section_ref)} ${esc(f.evidence_text).slice(0,200)}</td><td>${esc(f.ask)}</td><td>${esc(f.fallback)}</td></tr>`).join("");
 const q=D.queue.filter(x=>x.contract_id==id);
 document.getElementById("queue").innerHTML="<tr><th>family</th><th>question</th><th>agent draft</th></tr>"+(q.length?q.map(x=>`<tr><td>${x.family}</td><td>${esc(x.question)}</td><td>${esc(x.draft&&x.draft.verdict)} / ${esc(x.draft&&x.draft.severity)}</td></tr>`).join(""):'<tr><td colspan="3" class="mut">none</td></tr>');
}
pick.onchange=()=>{fam=null;render()};render();
</script></body></html>"""


def write_viewer(run_dir):
    run_dir = Path(run_dir)
    with open(run_dir / "results.csv", newline="") as f:
        rows = list(csv.DictReader(f))
    data = {
        "rows": rows,
        "ids": list(dict.fromkeys(r["contract_id"] for r in rows)),
        "trace": [json.loads(l) for l in open(run_dir / "trace.jsonl")],
        "queue": json.loads((run_dir / "human_queue.json").read_text()),
        "plans": json.loads((run_dir / "revision_plans.json").read_text()),
        "summary": json.loads((run_dir / "summary.json").read_text()),
    }
    blob = json.dumps(data, ensure_ascii=False, default=str).replace("</", "<\\/")
    out = run_dir / "trace_viewer.html"
    out.write_text(PAGE.replace("__DATA__", blob), encoding="utf-8")
    return out

## 14. Runner

In [ ]:
import csv
from pathlib import Path

COLUMNS = ["contract_id", "short_name", "split", "family", "verdict", "severity", "evidence_text",
           "section_ref", "evidence_2", "section_ref_2", "notes",
           "confidence", "path", "steps", "evidence_verified",
           "prepass_verdict", "prepass_severity", "prepass_confidence", "verdict_changed", "triggers_fired",
           "evidence_spans"]


def load_meta():
    with open(DATA_DIR / "contracts.csv", newline="") as f:
        return {r["contract_id"]: r for r in csv.DictReader(f)}


async def run_agent(ids, model, out_dir="runs/latest", embedder=None, condition="adaptive", allow_test=False):
    """Review contracts `ids` under one condition ("rag", "single_pass" or "adaptive");
    write results.csv, trace.jsonl, human_queue.json, revision_plans.json, summary.json and the viewer."""
    meta = load_meta()
    test_ids = [i for i in ids if meta[i]["split"] == "test"]
    if test_ids and not allow_test:
        raise ValueError(f"{test_ids} are test contracts. Tune on dev only; pass allow_test=True for the final scoring run.")
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    rows, traces, queue, plans, summary = [], [], [], {}, []
    for cid in ids:
        r = await review_contract(Contract.load(DATA_DIR / "contracts" / f"{cid}.txt", meta[cid]), model, embedder,
                                  condition)
        for fam in FAMILIES:
            f = r["results"][fam]
            rows.append({"contract_id": cid, "short_name": meta[cid]["short_name"], "split": meta[cid]["split"],
                         "family": fam, **{k: f.get(k, "") for k in COLUMNS[4:11]},
                         "confidence": f.get("confidence"), "path": f.get("path"), "steps": f.get("steps", 0),
                         "evidence_verified": f["evidence_text_verified"] and f["evidence_2_verified"],
                         "prepass_verdict": f["prepass"].get("verdict"), "prepass_severity": f["prepass"].get("severity"),
                         "prepass_confidence": f["prepass"].get("confidence"),
                         "verdict_changed": bool(f["prepass"]) and f["prepass"].get("verdict") != f.get("verdict"),
                         "triggers_fired": ", ".join(sorted(f["triggers"])),
                         "evidence_spans": json.dumps(f["evidence_spans"])})
        traces += r["trace"]
        queue += r["human_queue"]
        plans[cid] = r["plan"]
        summary.append(r["summary"])
        print(r["summary"])
    with open(out_dir / "results.csv", "w", newline="") as f:
        w = csv.DictWriter(f, COLUMNS)
        w.writeheader()
        w.writerows(rows)
    with open(out_dir / "trace.jsonl", "w") as f:
        for t in traces:
            f.write(json.dumps(t, ensure_ascii=False, default=list) + "\n")
    (out_dir / "human_queue.json").write_text(json.dumps(queue, indent=2, ensure_ascii=False, default=list))
    (out_dir / "revision_plans.json").write_text(json.dumps(plans, indent=2, ensure_ascii=False))
    (out_dir / "summary.json").write_text(json.dumps({"model": str(getattr(model, "model", model)), "condition": condition,
                                                      "contracts": summary, "bounds": vars(BOUNDS)}, indent=2))
    write_viewer(out_dir)
    print("wrote", out_dir)
    return out_dir

## 15. Free checks with a scripted fake model

`FakeAdkLlm` returns tool calls without calling any API, so this checks the ADK wiring, the step log, the no-progress detector, the step limit and the global tool cap. Run it after any change to the tools or callbacks.

In [ ]:
import re

from google.adk.models.base_llm import BaseLlm


class FakeAdkLlm(BaseLlm):
    """Scripted ADK model for free tests: exercises tools, the quote check, and the no-progress detector."""
    model: str = "fake"

    async def generate_content_async(self, llm_request, stream=False):
        sys_text = str(llm_request.config.system_instruction or "")
        first = "".join(p.text or "" for p in llm_request.contents[0].parts if p.text) if llm_request.contents else ""
        ctx = json.loads(first[first.index("{"):]) if "{" in first else {}
        usage = types.GenerateContentResponseUsageMetadata(prompt_token_count=1000, candidates_token_count=100)
        if "quick first pass" in sys_text:
            out = {}
            for i, (fam, ch) in enumerate(ctx.get("candidates", {}).items()):
                q = re.split(r"(?<=[.;])\s", ch[0]["text"])[0][:300] if ch else ""
                out[fam] = {"verdict": ["risky", "acceptable", "unclear"][i % 3], "severity": "medium",
                            "evidence_text": q, "section_ref": ch[0]["section"] if ch else "",
                            "confidence": [0.5, 0.9, 0.6, 0.95, 0.4, 0.8][i], "spans_conflict": i == 4, "notes": "fake"}
            text = json.dumps({"families": out})
        elif "Baseline:" in sys_text:
            ps = ctx.get("passages", [])
            q = re.split(r"(?<=[.;])\s", ps[0]["text"])[0][:300] if ps else ""
            text = json.dumps({"verdict": "risky", "severity": "medium", "evidence_text": q,
                               "section_ref": ps[0]["section"] if ps else "", "notes": "fake rag"})
        elif "negotiation ask" in sys_text:
            text = json.dumps({"items": [{"id": f["id"], "ask": "fake ask", "fallback": "fake fallback"}
                                         for f in ctx.get("findings", [])]})
        else:
            calls = [p.function_call for c in llm_request.contents for p in c.parts if p.function_call]
            n, fam = len(calls), ctx.get("family", "")
            trig, ev = ctx.get("open_triggers", {}), ctx.get("evidence", [])
            only_finalize = set(llm_request.tools_dict) == {"finalize"}
            if fam == "confidentiality" and not only_finalize:      # never finishes on its own
                name, args = "search_contract", {"query": f"confidential {n}", "mode": "sparse"}
            elif n == 0 and trig.get("cross_refs") and not only_finalize:
                name, args = "check_cross_reference", {"section_id": trig["cross_refs"][0]}
            elif n == 0 and not only_finalize:
                name, args = "search_contract", {"query": fam.replace("_", " "), "mode": "hybrid"}
            elif n == 1 and fam == "renewal" and not only_finalize:
                name, args = calls[0].name, dict(calls[0].args)            # deliberate repeat
            else:
                q = ev[0]["text"] if ev else ""
                name, args = "finalize", {"verdict": ctx["draft"].get("verdict", "unclear"), "severity": "low",
                                          "evidence_text": q, "section_ref": ev[0]["section"] if ev else "",
                                          "evidence_2": "", "section_ref_2": "", "notes": "fake finalize"}
            args = {k: v for k, v in args.items() if k not in ("rationale", "confidence")}
            if fam == "liability_indemnity" and n == 0:   # sloppy model: capitalised key, no confidence
                args |= {"Rationale": "fake step 1"}
            else:
                args |= {"rationale": f"fake step {n + 1}", "confidence": min(0.95, 0.5 + 0.15 * (n + 1))}
            yield LlmResponse(content=types.Content(role="model", parts=[types.Part.from_function_call(name=name, args=args)]),
                              usage_metadata=usage)
            return
        yield LlmResponse(content=types.Content(role="model", parts=[types.Part(text=text)]), usage_metadata=usage)

In [ ]:
import csv, json
DEV = ["C01", "C02", "C11", "C12"]

fake_dir = await run_agent(DEV, FakeAdkLlm(), "runs/adk_fake_dev")
tr = [json.loads(l) for l in open(fake_dir / "trace.jsonl")]
keys = {"step", "trigger_reason", "action", "rationale", "evidence_before", "evidence_after",
        "confidence_before", "confidence_after", "cost", "elapsed"}
assert all(keys <= set(t) for t in tr), "every step must be logged with the full record"
assert any(str(t.get("forced", "")).startswith("no progress") for t in tr)

saved = BOUNDS.max_steps_per_clause, BOUNDS.max_tool_calls_per_contract
BOUNDS.max_steps_per_clause = 2
d = await run_agent(["C01"], FakeAdkLlm(), "runs/check_steps")
assert max(json.loads(l)["step"] for l in open(d / "trace.jsonl")) <= 2
BOUNDS.max_steps_per_clause, BOUNDS.max_tool_calls_per_contract = 4, 3
d = await run_agent(["C01"], FakeAdkLlm(), "runs/check_cap")
assert any(json.loads(l).get("forced") == "global tool-call cap" for l in open(d / "trace.jsonl"))
BOUNDS.max_steps_per_clause, BOUNDS.max_tool_calls_per_contract = saved
for cond in ("rag", "single_pass"):
    d = await run_agent(["C01"], FakeAdkLlm(), f"runs/check_{cond}", condition=cond)
    assert all(r["path"] == ("rag" if cond == "rag" else "prepass") for r in csv.DictReader(open(d / "results.csv")))
print("all checks passed")

## 16. Real run on the dev contracts

Start with one contract to check the connection and cost. Then run the three conditions on all four dev contracts; each goes to its own folder.

In [ ]:
out_dir = await run_agent(["C01"], ADK_MODEL, "runs/adk_C01", embedder=EMBEDDER)

In [ ]:
RUNS = {}
for cond in ("rag", "single_pass", "adaptive"):
    d = Path(f"runs/dev_{cond}")
    if (d / "results.csv").exists():      # already finished: skip it, so a rerun after an error costs nothing
        print("skip", cond, "(delete the folder to rerun it)")
        RUNS[cond] = d
        continue
    RUNS[cond] = await run_agent(DEV, ADK_MODEL, str(d), embedder=EMBEDDER, condition=cond)
out_dir = RUNS["adaptive"]

In [ ]:
# Quick look: how often did the follow-up loop change the pre-pass verdict, and what did it cost?
import pandas as pd
full = pd.read_csv(RUNS["adaptive"] / "results.csv")
looped = full[full.path != "prepass"]
print("families that entered the loop:", len(looped), "/", len(full))
print("verdict changed by the loop:", int(looped.verdict_changed.sum()))
display(pd.crosstab(looped.prepass_verdict, looped.verdict, margins=True))
for cond, d in RUNS.items():
    s = pd.DataFrame(json.load(open(d / "summary.json"))["contracts"])
    print(f"{cond:12s} cost ${s.cost_usd.sum():.4f}  time {s.elapsed_s.sum():.0f}s  tool calls {s.tool_calls.sum()}")

In [ ]:
import pandas as pd
res = pd.read_csv(out_dir / "results.csv")
display(pd.DataFrame(json.load(open(out_dir / "summary.json"))["contracts"]))
res[["contract_id", "family", "verdict", "severity", "confidence", "path", "steps", "evidence_verified", "section_ref"]]

## 17. Trace viewer

Click a family row to see each step. Forced stops are highlighted. Download `trace_viewer.html` to open it in a browser.

In [ ]:
from IPython.display import HTML, display
display(HTML((out_dir / "trace_viewer.html").read_text()))

## 18. Sanity check against the draft labels (dev only)

`Jenny_Labels_claude.xlsx` is an **AI draft**, not the gold standard. Use it only to spot obvious problems while tuning.

In [ ]:
draft = pd.read_excel(DATA_DIR / "Jenny_Labels_claude.xlsx", sheet_name="labels")[["contract_id", "family", "verdict", "severity"]]
cmp = res.merge(draft, on=["contract_id", "family"], suffixes=("_agent", "_draft"))
print("verdict agreement with draft:", round((cmp.verdict_agent == cmp.verdict_draft).mean(), 2))
cmp[["contract_id", "family", "verdict_agent", "verdict_draft", "severity_agent", "severity_draft", "path"]]

## 19. Questions queued for a human

In [ ]:
pd.DataFrame(json.load(open(out_dir / "human_queue.json")))

## 20. Download the run

In [ ]:
if IN_COLAB:
    !zip -qr run.zip {out_dir}
    from google.colab import files
    files.download("run.zip")